In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve
)

from src.data_loader import load_elliptic_data
from src.models.hybrid import HybridModel
from src.utils import create_masks, set_seed

In [ ]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
data = load_elliptic_data()
data = create_masks(data)

data = data.to(device)

print("Nodes:", data.num_nodes)
print("Edges:", data.num_edges)

In [ ]:
model = HybridModel(input_dim=data.x.shape[1]).to(device)
model.load_state_dict(torch.load("best_model.pth", map_location=device))

model.eval()
print("Model loaded successfully")

In [ ]:
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

y_true = data.y[data.test_mask].cpu()
y_pred = pred[data.test_mask].cpu()

probs = torch.exp(out[data.test_mask])[:, 1].cpu()

In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_true, probs)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr)
plt.title(f"ROC Curve (AUC = {roc_auc:.3f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_true, probs)

plt.figure()
plt.plot(recall, precision)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

In [ ]:
scores = torch.exp(out)[:, 1].cpu().numpy()

plt.figure()
plt.hist(scores, bins=50)
plt.title("Fraud Risk Score Distribution")
plt.xlabel("Risk Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
labels = data.y.cpu().numpy()

plt.figure()
plt.hist(labels, bins=2)
plt.title("Class Distribution (Licit vs Illicit)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

accuracy = (y_pred == y_true).sum().item() / len(y_true)
print("Test Accuracy:", accuracy)